# STEP 2. 테이블 결합

STEP 1 에서 이미 10개 업종 · 13분기로 좁혀둔 5개 테이블을 상권 단위로 붙입니다.

## 결합 키

| 오른쪽 테이블 | 키 | 관계 |
|---|---|---|
| 영역-상권 | `상권_코드` | 시간에 안 변하므로 분기 없음 |
| 길단위인구 | `상권_코드` + `분기` | |
| 집객시설 | `상권_코드` + `분기` | |
| 추정매출 | `상권_코드` + `분기` + `업종` | |

## 이 단계 최대 위험 — 행 증식

오른쪽 테이블에 키가 중복되어 있으면 좌결합만으로 행이 늘어납니다. 어떤 상권-분기가 길단위인구에 두 번 들어 있으면 그 상권의 **모든 업종 행이 두 배**가 됩니다. 나중에 발견하면 원인 추적이 매우 어렵습니다.

방어는 두 겹입니다.

1. **사전** — 결합 전에 오른쪽 테이블 키의 유일성을 직접 확인
2. **사후** — `validate=` 인자와 행 수 비교로 이중 확인

## STEP 1 개편 반영

- 점포 총수 컬럼은 **`전체_점포_수`** 입니다 (`점포_수` 아님)
- `상권_전체점포` · `상권_외식점포` · `외식_비중` · `프랜차이즈_비율` 은 **STEP 1 에서 이미 계산**했습니다. 여기서 다시 만들지 않습니다
- 요식업 필터도 STEP 1 에서 끝났습니다


In [1]:
import sys, os
from pathlib import Path

# 노트북이 어디서 열리든 프로젝트 루트를 찾아 sys.path에 추가
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
print("프로젝트 루트:", ROOT)

프로젝트 루트: c:\Users\spide\ai-data-bootcamp\project\h2_nb


In [2]:
from config import PROC, QUARTERS, FOOD, MIN_STORES

store    = pd.read_pickle(PROC / "01_store.pkl")
sales    = pd.read_pickle(PROC / "01_sales.pkl")
area     = pd.read_pickle(PROC / "01_area.pkl")
flow     = pd.read_pickle(PROC / "01_flow.pkl")
facility = pd.read_pickle(PROC / "01_facility.pkl")

for nm, d in [("store", store), ("sales", sales), ("area", area),
              ("flow", flow), ("facility", facility)]:
    print(f"  {nm:9} {str(d.shape):>14}")

assert "전체_점포_수" in store.columns, "STEP 1 을 새 버전으로 다시 실행하세요"
assert store["서비스_업종_코드_명"].nunique() == len(FOOD), "업종 필터가 안 되어 있음"
print(f"\n분기 {sorted(store['기준_년분기_코드'].unique())}")

  store       (160352, 15)
  sales         (87558, 5)
  area          (1650, 10)
  flow         (21434, 24)
  facility     (20514, 22)

분기 [np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234), np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20244), np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254), np.int64(20261)]


## 2-1. 결합 전 키 유일성 검사

행 증식은 여기서 막는 게 정석입니다. 사후 행 수 비교는 백업일 뿐입니다.

In [3]:
def check_key(df, keys, name):
    dup = df.duplicated(keys).sum()
    status = "OK" if dup == 0 else f"❌ 중복 {dup:,}"
    print(f"  {name:12} {'+'.join(keys):40} {status}")
    assert dup == 0, f"{name} 키 중복 — 결합하면 행이 증식합니다"

print("[오른쪽 테이블 키 유일성]")
check_key(area,     ["상권_코드"],                                  "영역-상권")
check_key(flow,     ["기준_년분기_코드", "상권_코드"],                "길단위인구")
check_key(facility, ["기준_년분기_코드", "상권_코드"],                "집객시설")
check_key(sales,    ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"], "추정매출")

print("\n[왼쪽 테이블 키 유일성]")
check_key(store, ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"], "점포(기준)")

[오른쪽 테이블 키 유일성]
  영역-상권        상권_코드                                    OK
  길단위인구        기준_년분기_코드+상권_코드                          OK
  집객시설         기준_년분기_코드+상권_코드                          OK
  추정매출         기준_년분기_코드+상권_코드+서비스_업종_코드                OK

[왼쪽 테이블 키 유일성]
  점포(기준)       기준_년분기_코드+상권_코드+서비스_업종_코드                OK


## 2-2. 좌결합

`validate="many_to_one"` 은 오른쪽 키가 유일하지 않으면 pandas 가 직접 예외를 던지게 합니다. 행 수 비교와 합쳐 두 겹으로 막습니다.

In [4]:
n0 = len(store)
df = store.copy()

def join(right, on, name, how="left", validate="many_to_one"):
    global df
    before = len(df)
    df = df.merge(right, on=on, how=how, validate=validate)
    assert len(df) == before == n0, f"[{name}] 행 증식 {before:,} → {len(df):,}"
    print(f"  [{name:10}] 행 유지 {len(df):,}  OK")

join(area[["상권_코드", "상권_구분_코드_명", "자치구_코드_명", "행정동_코드_명",
           "영역_면적", "엑스좌표_값", "와이좌표_값"]]
     .rename(columns={"상권_구분_코드_명": "상권유형",
                      "자치구_코드_명": "자치구",
                      "행정동_코드_명": "행정동"}),
     "상권_코드", "영역-상권")

join(flow[["기준_년분기_코드", "상권_코드", "총_유동인구_수"]],
     ["기준_년분기_코드", "상권_코드"], "길단위인구")

join(facility[["기준_년분기_코드", "상권_코드", "집객시설_수", "지하철_역_수"]],
     ["기준_년분기_코드", "상권_코드"], "집객시설")

join(sales, ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"],
     "추정매출", validate="one_to_one")

# 점포 테이블의 상권_구분_코드_명과 영역-상권의 상권유형이 일치하는지 교차 확인
mismatch = (df["상권_구분_코드_명"] != df["상권유형"]).sum()
print(f"\n상권유형 출처 두 곳 불일치 : {mismatch:,}건")
df = df.drop(columns=["상권_구분_코드_명"])
print(f"결합 완료 {df.shape}")

  [영역-상권     ] 행 유지 160,352  OK
  [길단위인구     ] 행 유지 160,352  OK
  [집객시설      ] 행 유지 160,352  OK
  [추정매출      ] 행 유지 160,352  OK

상권유형 출처 두 곳 불일치 : 0건
결합 완료 (160352, 25)


## 2-3. 결측 진단

In [5]:
rows = []
for c in ["자치구", "영역_면적", "엑스좌표_값", "총_유동인구_수",
          "집객시설_수", "지하철_역_수", "당월_매출_금액", "상권_전체점포"]:
    r = df[c].isna().mean()
    판정 = "그대로 사용" if r < 0.05 else ("주의" if r < 0.30 else "매칭 공변량 제외")
    rows.append({"컬럼": c, "결측률": f"{r:.1%}", "판정": 판정})
display(pd.DataFrame(rows))

,컬럼,결측률,판정
0,자치구,0.0%,그대로 사용
1,영역_면적,0.0%,그대로 사용
2,엑스좌표_값,0.0%,그대로 사용
3,총_유동인구_수,0.0%,그대로 사용
4,집객시설_수,2.3%,그대로 사용
5,지하철_역_수,85.0%,매칭 공변량 제외
6,당월_매출_금액,45.4%,매칭 공변량 제외
7,상권_전체점포,0.0%,그대로 사용


## 2-4. 매출 결측의 정체 ★

`당월_매출_금액` 결측 45% 는 이전에 "매출 파일이 일부 업종만 커버해서"라고 적어두었지만 **사실이 아닙니다.** 매출 파일에는 10개 외식업이 모두 들어 있습니다.

진짜 원인은 **칸 단위 비식별 처리**입니다. 점포가 몇 개 없는 상권-업종 칸은 매출을 공개하면 개별 점포 매출이 드러나므로 값을 비워둡니다. 아래 표가 그 구조를 보여줍니다.

In [6]:
df["매출결측"] = df["당월_매출_금액"].isna()
bins = pd.cut(df["전체_점포_수"], [-1, 0, 1, 2, 4, 9, 19, 49, 10**6],
              labels=["0", "1", "2", "3-4", "5-9", "10-19", "20-49", "50+"])
t = (df.groupby(bins, observed=True)
     .agg(칸수=("매출결측", "size"), 매출결측률=("매출결측", "mean")))
display(t.round(3))

sub = df[df["전체_점포_수"] >= MIN_STORES]
print(f"전체                    매출결측 {df['매출결측'].mean():.1%}")
print(f"점포 {MIN_STORES}개 이상 칸만       매출결측 {sub['매출결측'].mean():.1%}")
print(f"점포 {MIN_STORES}개 이상 + 골목상권  매출결측 "
      f"{sub[sub['상권유형']=='골목상권']['매출결측'].mean():.1%}")
print("\n→ STEP 3 이 어차피 점포 5개 미만을 걸러내므로,")
print("   분석 표본에서 매출은 90% 이상 관측됩니다. '매출은 못 쓴다'는 판단은 재검토 대상.")

,칸수,매출결측률
전체_점포_수,,
0,1134,1.000
1,33812,1.000
2,22715,1.000
3-4,29766,0.335
5-9,31916,0.135
10-19,20472,0.033
20-49,14310,0.013
50+,6227,0.002


전체                    매출결측 45.4%
점포 5개 이상 칸만       매출결측 7.1%
점포 5개 이상 + 골목상권  매출결측 9.1%

→ STEP 3 이 어차피 점포 5개 미만을 걸러내므로,
   분석 표본에서 매출은 90% 이상 관측됩니다. '매출은 못 쓴다'는 판단은 재검토 대상.


In [7]:
print("[점포 5개 이상 칸] 업종별 매출 결측률")
display(sub.groupby("서비스_업종_코드_명")["매출결측"]
        .mean().sort_values().round(3).rename("매출결측률").to_frame())
print("※ 결측이 업종마다 다릅니다. 매출을 매칭 공변량에 넣으면")
print("   업종별로 비대칭하게 표본이 빠지므로, 주 분석이 아니라 강건성 분석으로 다뤄야 합니다.")
df = df.drop(columns=["매출결측"])

[점포 5개 이상 칸] 업종별 매출 결측률


,매출결측률
서비스_업종_코드_명,
한식음식점,0.012
호프-간이주점,0.014
분식전문점,0.053
중식음식점,0.061
커피-음료,0.073
치킨전문점,0.090
일식음식점,0.126
패스트푸드점,0.158
제과점,0.162


※ 결측이 업종마다 다릅니다. 매출을 매칭 공변량에 넣으면
   업종별로 비대칭하게 표본이 빠지므로, 주 분석이 아니라 강건성 분석으로 다뤄야 합니다.


## 2-5. 분기별 커버리지

13분기가 모두 붙었는지, 결측이 특정 분기에 몰려 있지 않은지 봅니다. 특정 분기만 튀면 원본 파일 문제입니다.

In [8]:
cov = df.groupby("기준_년분기_코드").agg(
    행수=("전체_점포_수", "size"),
    상권수=("상권_코드", "nunique"),
    유동인구결측=("총_유동인구_수", lambda s: s.isna().mean()),
    집객시설결측=("집객시설_수", lambda s: s.isna().mean()),
    매출결측=("당월_매출_금액", lambda s: s.isna().mean()))
display(cov.round(3))

assert set(cov.index) == set(QUARTERS), "분기 커버리지가 config.QUARTERS 와 다릅니다"
print(f"분기 {len(cov)}개 모두 존재 ✅")
display(df["상권유형"].value_counts().to_frame("행 수"))

,행수,상권수,유동인구결측,집객시설결측,매출결측
기준_년분기_코드,,,,,
20231,12397,1631,0.000,0.023,0.446
20232,12409,1631,0.000,0.023,0.448
20233,12393,1630,0.000,0.023,0.447
20234,12405,1631,0.000,0.023,0.448
20241,12418,1632,0.000,0.023,0.450
20242,12388,1632,0.000,0.023,0.452
20243,12348,1633,0.001,0.023,0.453
20244,12320,1633,0.000,0.023,0.455
20251,12304,1633,0.000,0.023,0.459


분기 13개 모두 존재 ✅


,행 수
상권유형,
골목상권,102767
발달상권,31378
전통시장,25435
관광특구,772


## 2-6. 저장

In [9]:
df.to_pickle(PROC / "02_merged.pkl")
mb = (PROC / "02_merged.pkl").stat().st_size / 1024**2
print(f"[저장] 02_merged.pkl  {df.shape}  {mb:.1f} MB")
print(f"컬럼 {len(df.columns)}개")
print(list(df.columns))

[저장] 02_merged.pkl  (160352, 25)  42.0 MB
컬럼 25개
['기준_년분기_코드', '상권_코드', '상권_코드_명', '서비스_업종_코드', '서비스_업종_코드_명', '전체_점포_수', '일반_점포_수', '프랜차이즈_점포_수', '개업_점포_수', '폐업_점포_수', '상권_전체점포', '상권_외식점포', '외식_비중', '프랜차이즈_비율', '상권유형', '자치구', '행정동', '영역_면적', '엑스좌표_값', '와이좌표_값', '총_유동인구_수', '집객시설_수', '지하철_역_수', '당월_매출_금액', '당월_매출_건수']


---

## STEP 3 에서 할 일

- `y(t) = [순증감(t+1) < 0]` — `shift(-1)` 은 (상권, 업종) 그룹 내 분기 정렬 후
- 20261 이 있으므로 **20254 까지 라벨이 붙습니다** → 학습 구간 12분기 (20231~20254)
- 20261 행 자체는 라벨을 못 만들어 탈락 — 정상입니다
- 필터: 라벨 존재 → `전체_점포_수` ≥ 5 → 유동인구 존재 → 면적 유효
- 점포 총수 기준이 바뀌었으므로 **5개 이상 필터를 통과하는 칸이 이전보다 늘어납니다** (프랜차이즈 밀집 업종이 특히)
